# 📓 Notebook 06 — Experiment 02: Improved CNN
## G19 Facial Emotion Recognition | CSE276
**Team:** Aryan Priyadarshi · Yuvraj · Mashwin Sharma

---

## 1. Experiment Overview

This notebook runs **Experiment 02** completely independently from **Experiment 01 (Baseline)**.

### Baseline Reference (Experiment 01 — Preserved Unchanged)
- **Architecture**: 3-block CNN (`src/model.py::build_cnn_model`)
- **Validation accuracy (`PublicTest`)**: `58.46%` (best epoch `40`)
- **`PrivateTest` accuracy / Macro F1 / Weighted F1**: `60.96%` / `0.5771` / `0.5948`
- **Weakest classes**: Fear (`0.3144`), Sad (`0.4252`), Angry (`0.5308`)

### Experiment 02 Controlled Ablation Plan
- **02-A**: 4-block Improved CNN (Head A) + Baseline augmentation + Full balanced class weights
- **02-B**: 4-block Improved CNN (Head A) + Refined augmentation + Full balanced class weights
- **02-C**: Best augmentation from A/B + Square-root-smoothed balanced class weights
- **02-D**: Best augmentation from A/B + No class weights (`None`)
- **Head Comparison**: Compare Classification Head A vs. Head B using the best augmentation & class-weight strategy

> **Strict Evaluation Rule:** Training uses `Training` (`28,709`). Model selection uses `PublicTest` (`3,589`) **only**, with **PublicTest Macro F1** as the primary selection metric. `PrivateTest` is **never** loaded, inspected, evaluated, or tuned here.

## 2. Environment Verification

In [ ]:
import os, sys, time, shutil
sys.path.insert(0, '/content/G19_Facial_Emotion_Recognition')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix, f1_score, precision_score, recall_score

from src.data_loader import (
    load_fer2013_train_val,
    compute_training_class_weights,
    compute_smoothed_class_weights,
    EMOTION_LABELS,
    NUM_CLASSES,
)
from src.preprocessing import build_augmentation_layer, build_refined_augmentation_layer
from src.model import build_improved_cnn_model, get_exp02_callbacks
from src.utils import set_random_seeds

set_random_seeds(42)

DATASET_CSV      = '/content/G19_Facial_Emotion_Recognition/dataset/fer2013.csv'
RESULTS_DIR      = '/content/G19_Facial_Emotion_Recognition/results'
MODELS_DIR       = '/content/G19_Facial_Emotion_Recognition/models'
DRIVE_MODELS     = '/content/drive/MyDrive/G19_FER/models'
DRIVE_RESULTS    = '/content/drive/MyDrive/G19_FER/results'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR,  exist_ok=True)

# Ensure Google Drive is mounted before creating Drive checkpoint paths
if not os.path.exists('/content/drive/MyDrive'):
    from google.colab import drive
    drive.mount('/content/drive')
os.makedirs(DRIVE_MODELS,  exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)

print(f'Python     : {sys.version.split()[0]}')
print(f'TensorFlow : {tf.__version__}')
print('Environment verified.')

## 3. GPU Verification

In [ ]:
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'✅ GPU detected: {gpus}')
else:
    print('⚠️  No GPU detected. Enable GPU via Runtime → Change runtime type → T4 GPU.')

## 4. Dataset Loading
Load `Training` and `PublicTest` only via `load_fer2013_train_val()`. `PrivateTest` rows are never decoded, materialized, or returned in this notebook.

In [ ]:
# PrivateTest rows are never decoded or returned during Experiment 02
X_train, y_train, X_val, y_val = load_fer2013_train_val(DATASET_CSV)
CLASS_NAMES = [EMOTION_LABELS[i] for i in range(NUM_CLASSES)]
print('Loaded Training and PublicTest splits only (PrivateTest untouched).')

## 5. Dataset Split Verification

In [ ]:
assert X_train.shape == (28709, 48, 48, 1), f'Unexpected X_train shape: {X_train.shape}'
assert X_val.shape   == (3589, 48, 48, 1),  f'Unexpected X_val shape: {X_val.shape}'
assert set(np.unique(y_train)) == set(range(7)), 'Labels must be 0..6'
assert set(np.unique(y_val))   == set(range(7)), 'Val labels must be 0..6'

print(f'Training   : X={X_train.shape} {X_train.dtype} range=[{X_train.min():.3f}, {X_train.max():.3f}]')
print(f'PublicTest : X={X_val.shape} {X_val.dtype} range=[{X_val.min():.3f}, {X_val.max():.3f}]')
print('✅ Split verification passed.')

## 6. Class-Weight Calculation
Compute both **full balanced** weights and **square-root-smoothed** weights exclusively from `y_train`.

In [ ]:
full_class_weights     = compute_training_class_weights(y_train)
smoothed_class_weights = compute_smoothed_class_weights(y_train, power=0.5)

print(f"{'Class':<12} {'Full Balanced':>14} {'Sqrt Smoothed':>14}")
print('-' * 42)
for i in range(NUM_CLASSES):
    print(f"{EMOTION_LABELS[i]:<12} {full_class_weights[i]:>14.4f} {smoothed_class_weights[i]:>14.4f}")

## 7. Augmentation Visualisation
Compare **Baseline augmentation** (`Flip`, `±10°`, `±10%` zoom, `±15%` brightness) vs. **Refined augmentation** (`Flip`, `±7°`, `±10%` zoom, `±8%` translation, `±15%` brightness, `±15%` contrast) on a training image. Validation images are never augmented.

In [ ]:
base_aug = build_augmentation_layer()
ref_aug  = build_refined_augmentation_layer()
sample   = X_train[0:1]

fig, axes = plt.subplots(2, 5, figsize=(13, 5))
for row, (title, aug) in enumerate([('Baseline Aug', base_aug), ('Refined Aug', ref_aug)]):
    axes[row, 0].imshow(sample[0, :, :, 0], cmap='gray', vmin=0, vmax=1)
    axes[row, 0].set_title(f'{title}\nOriginal', fontsize=9)
    axes[row, 0].axis('off')
    for col in range(1, 5):
        aug_img = aug(sample, training=True).numpy()
        axes[row, col].imshow(np.clip(aug_img[0, :, :, 0], 0, 1), cmap='gray', vmin=0, vmax=1)
        axes[row, col].set_title(f'Sample {col}', fontsize=9)
        axes[row, col].axis('off')

plt.suptitle(f'Experiment 02 Augmentation Comparison ({EMOTION_LABELS[y_train[0]]})', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/exp02_augmentation_comparison.png', dpi=120)
plt.show()

## 8. Improved CNN Construction & Experiment Runner
Verify `build_improved_cnn_model()` for both **Head A** (`Flatten`) and **Head B** (`GAP + GMP`), and define a clean helper to run each controlled experiment and record all `PublicTest` metrics.

In [ ]:
for head in ('A', 'B'):
    tmp_model = build_improved_cnn_model(head_type=head, augmentation_type='baseline')
    print(f'Head {head} — {tmp_model.name}: {tmp_model.count_params():,} params')
    del tmp_model
tf.keras.backend.clear_session()

BATCH_SIZE = 64
MAX_EPOCHS = 60
ablation_records = []

def run_exp02_arm(
    run_id: str,
    augmentation_type: str,
    class_weight_mode: str,
    class_weights_dict: dict | None,
    head_type: str = 'A',
) -> dict:
    """Train one Experiment 02 arm on Training and evaluate on PublicTest only."""
    set_random_seeds(42)
    tf.keras.backend.clear_session()

    local_ckpt = f'{MODELS_DIR}/exp02_{run_id}.keras'
    drive_ckpt = f'{DRIVE_MODELS}/exp02_{run_id}_drive.keras'

    model = build_improved_cnn_model(
        input_shape=(48, 48, 1),
        num_classes=NUM_CLASSES,
        use_augmentation=True,
        augmentation_type=augmentation_type,
        head_type=head_type,
        learning_rate=1e-3,
    )
    callbacks = get_exp02_callbacks(local_save_path=local_ckpt, drive_save_path=drive_ckpt)

    t0 = time.perf_counter()
    history = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=MAX_EPOCHS,
        validation_data=(X_val, y_val),
        class_weight=class_weights_dict,
        callbacks=callbacks,
        verbose=1,
    )
    duration_sec = time.perf_counter() - t0

    # Save training history CSV
    hist_df = pd.DataFrame(history.history)
    hist_df.insert(0, 'epoch', range(1, len(hist_df) + 1))
    hist_df.to_csv(f'{RESULTS_DIR}/exp02_training_history_{run_id}.csv', index=False)

    # Save training curves PNG
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
    ax1.plot(hist_df['epoch'], hist_df['loss'], label='Train Loss')
    ax1.plot(hist_df['epoch'], hist_df['val_loss'], label='PublicTest Val Loss')
    ax1.set_title(f'{run_id} — Loss'); ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(hist_df['epoch'], hist_df['accuracy'], label='Train Acc')
    ax2.plot(hist_df['epoch'], hist_df['val_accuracy'], label='PublicTest Val Acc')
    ax2.set_title(f'{run_id} — Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/exp02_training_curves_{run_id}.png', dpi=150)
    plt.close(fig)

    # Evaluate best restored weights strictly on PublicTest
    val_loss, val_acc = model.evaluate(X_val, y_val, verbose=0)
    y_val_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)

    macro_p = precision_score(y_val, y_val_pred, average='macro', zero_division=0)
    macro_r = recall_score(y_val, y_val_pred, average='macro', zero_division=0)
    macro_f1 = f1_score(y_val, y_val_pred, average='macro', zero_division=0)
    weighted_f1 = f1_score(y_val, y_val_pred, average='weighted', zero_division=0)

    # Save PublicTest classification report CSV
    rep_dict = classification_report(
        y_val, y_val_pred, labels=list(range(NUM_CLASSES)),
        target_names=CLASS_NAMES, output_dict=True, zero_division=0
    )
    pd.DataFrame(rep_dict).transpose().to_csv(f'{RESULTS_DIR}/exp02_classification_report_{run_id}.csv')

    # Save PublicTest confusion matrix PNG
    cm = confusion_matrix(y_val, y_val_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_title(f'PublicTest Confusion Matrix — {run_id}')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    plt.savefig(f'{RESULTS_DIR}/exp02_confusion_matrix_{run_id}.png', dpi=150)
    plt.close(fig)

    best_epoch = int(hist_df.loc[hist_df['val_loss'].idxmin(), 'epoch'])
    record = {
        'run_id'           : run_id,
        'head_type'        : head_type,
        'augmentation'     : augmentation_type,
        'class_weights'    : class_weight_mode,
        'epochs_run'       : len(hist_df),
        'best_epoch'       : best_epoch,
        'duration_sec'     : round(duration_sec, 1),
        'val_loss'         : round(float(val_loss), 4),
        'val_accuracy'     : round(float(val_acc), 4),
        'macro_precision'  : round(float(macro_p), 4),
        'macro_recall'     : round(float(macro_r), 4),
        'macro_f1'         : round(float(macro_f1), 4),
        'weighted_f1'      : round(float(weighted_f1), 4),
        'local_checkpoint' : local_ckpt,
        'drive_checkpoint' : drive_ckpt,
    }
    print(f"\n✅ {run_id} finished: Val Acc={val_acc*100:.2f}% | Macro F1={macro_f1:.4f} | Val Loss={val_loss:.4f}")
    return record

## 9. Experiment 02-A (Improved CNN + Baseline Augmentation + Full Balanced Weights)

In [ ]:
res_02a = run_exp02_arm(
    run_id='02A_base_aug_full_cw',
    augmentation_type='baseline',
    class_weight_mode='full_balanced',
    class_weights_dict=full_class_weights,
    head_type='A',
)
ablation_records.append(res_02a)

## 10. Experiment 02-B (Improved CNN + Refined Augmentation + Full Balanced Weights)

In [ ]:
res_02b = run_exp02_arm(
    run_id='02B_refined_aug_full_cw',
    augmentation_type='refined',
    class_weight_mode='full_balanced',
    class_weights_dict=full_class_weights,
    head_type='A',
)
ablation_records.append(res_02b)

# Select best augmentation between 02-A and 02-B by PublicTest Macro F1
best_aug_type = 'refined' if res_02b['macro_f1'] >= res_02a['macro_f1'] else 'baseline'
print(f'Selected augmentation for 02-C and 02-D: {best_aug_type}')

## 11. Experiment 02-C (Best Augmentation + Square-Root-Smoothed Class Weights)

In [ ]:
res_02c = run_exp02_arm(
    run_id=f'02C_{best_aug_type}_aug_sqrt_cw',
    augmentation_type=best_aug_type,
    class_weight_mode='sqrt_smoothed',
    class_weights_dict=smoothed_class_weights,
    head_type='A',
)
ablation_records.append(res_02c)

## 12. Experiment 02-D (Best Augmentation + No Class Weights)

In [ ]:
res_02d = run_exp02_arm(
    run_id=f'02D_{best_aug_type}_aug_no_cw',
    augmentation_type=best_aug_type,
    class_weight_mode='none',
    class_weights_dict=None,
    head_type='A',
)
ablation_records.append(res_02d)

## 13. Classification Head Comparison (Head A vs. Head B)
Using the best augmentation and class-weight strategy from 02-A–02-D (by PublicTest Macro F1), train **Head B** (`GAP + GMP`) and compare against the winning Head A run.

In [ ]:
best_ad = max(ablation_records, key=lambda r: (r['macro_f1'], -r['val_loss'], r['val_accuracy']))
cw_lookup = {
    'full_balanced': full_class_weights,
    'sqrt_smoothed': smoothed_class_weights,
    'none'         : None,
}

print(f"Best Head A config: {best_ad['run_id']} (Macro F1={best_ad['macro_f1']:.4f})")
res_head_b = run_exp02_arm(
    run_id=f"02E_headB_{best_ad['augmentation']}_{best_ad['class_weights']}",
    augmentation_type=best_ad['augmentation'],
    class_weight_mode=best_ad['class_weights'],
    class_weights_dict=cw_lookup[best_ad['class_weights']],
    head_type='B',
)
ablation_records.append(res_head_b)

## 14. Results Comparison

In [ ]:
summary_df = pd.DataFrame(ablation_records).sort_values(
    by=['macro_f1', 'val_loss', 'val_accuracy'],
    ascending=[False, True, False],
).reset_index(drop=True)

summary_csv = f'{RESULTS_DIR}/exp02_ablation_summary.csv'
summary_df.to_csv(summary_csv, index=False)
print(f'Saved ablation summary → {summary_csv}')
display(summary_df[['run_id', 'head_type', 'augmentation', 'class_weights', 'best_epoch', 'val_loss', 'val_accuracy', 'macro_precision', 'macro_recall', 'macro_f1', 'weighted_f1']])

## 15. Best Configuration Selection

In [ ]:
winner = summary_df.iloc[0]
print('🏆 Best Experiment 02 Configuration (by PublicTest Macro F1):')
for col in ['run_id', 'head_type', 'augmentation', 'class_weights', 'best_epoch', 'val_loss', 'val_accuracy', 'macro_f1', 'weighted_f1']:
    print(f'  {col:<16}: {winner[col]}')

## 16. Save Final Selected Checkpoint

In [ ]:
final_local = f'{MODELS_DIR}/exp02_best_model.keras'
final_drive = f'{DRIVE_MODELS}/exp02_best_model_drive.keras'

shutil.copy(winner['local_checkpoint'], final_local)
shutil.copy(winner['local_checkpoint'], final_drive)
shutil.copy(summary_csv, f'{DRIVE_RESULTS}/exp02_ablation_summary.csv')

print(f'✅ Final selected model saved locally : {final_local}')
print(f'✅ Final selected model saved to Drive: {final_drive}')
print('⚠️  Next step (separate approval): evaluate this winning model ONCE on PrivateTest.')